# W78 — Round-robin evaluation

Notebook này không chứa bản sao agent hay thuật toán. Runner/analyzer canonical nằm ở `scripts/`; raw JSONL là nguồn dữ liệu duy nhất cho mọi bảng kết quả.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "scripts" / "run_tournament_w78.py").is_file():
    ROOT = ROOT.parent
assert (ROOT / "scripts" / "run_tournament_w78.py").is_file(), "Open from Caro_AI_Project or notebooks/"
CONFIG = ROOT / "configs" / "tournament_w78.json"
RESULTS = ROOT / "results" / "w78"
config = json.loads(CONFIG.read_text(encoding="utf-8"))
pd.DataFrame([{
    "Checkpoint": config["checkpoint"],
    "Ván/cặp/profile": config["games_per_pair"],
    "Profile": ", ".join(config["profiles"]),
    "Agent": ", ".join(config["agents"]),
}])

,Checkpoint,Ván/cặp/profile,Profile,Agent
0,models/checkpoints/model-v5-natural-teachers-r...,100,"DEFAULT, FAIR","random, heuristic, minimax, mcts, neural"


## Chạy chính thức

Chỉ chạy sau khi đã freeze code/model/config. Lệnh có resume theo `game_id`; không sửa config hoặc checkpoint giữa chừng.

In [2]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/run_tournament_w78.py', '--profile', 'ALL', '--results-dir', 'results/w78'], cwd=ROOT, check=True)
subprocess.run([sys.executable, 'scripts/analyze_results_w78.py', '--results-dir', 'results/w78', '--profile', 'ALL'], cwd=ROOT, check=True)

CompletedProcess(args=['C:\\Users\\pc\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', 'scripts/analyze_results_w78.py', '--results-dir', 'results/w78', '--profile', 'ALL'], returncode=0)

## Đọc bảng tái sinh

Sau khi analyzer hoàn tất, cell dưới hiển thị ma trận đối đầu, W/D/L, Wilson 95% CI, latency, nodes, simulations và seed sensitivity.

In [3]:
analysis_path = RESULTS / "analysis.json"
if not analysis_path.is_file():
    raise FileNotFoundError("Chưa có analysis.json. Hãy chạy analyzer canonical trước.")
analysis = json.loads(analysis_path.read_text(encoding="utf-8"))

def percent(value):
    return "—" if value is None else f"{value * 100:.2f}%"

def wdl(stats):
    return f"{stats['wins']} / {stats['draws']} / {stats['losses']}"

for profile, values in analysis["profiles"].items():
    display(Markdown(f"## {profile} — {values['games']} ván, {len(values['pair_matrix'])} cặp, {values['budget_violations']} budget violation"))

    overview = []
    for agent, item in values["agents"].items():
        stats = item["overall"]
        ci = stats["win_ci95"]
        overview.append({
            "Agent": agent,
            "W / D / L": wdl(stats),
            "Score": percent(stats["score_rate"]),
            "Win CI 95%": "—" if ci is None else f"{ci[0] * 100:.2f}–{ci[1] * 100:.2f}%",
            "Mean move (ms)": round(stats["mean_move_ms"], 2),
            "P95 move (ms)": round(stats["p95_move_ms"], 2),
            "Nodes/move": round(stats["mean_nodes_per_move"], 2),
            "Sims/move": round(stats["mean_simulations_per_move"], 2),
            "Seed score SD": round(item["seed_sensitivity"]["score_sample_stdev"], 3),
        })
    display(Markdown("### Tổng hợp agent"))
    display(pd.DataFrame(overview).sort_values("Score", ascending=False, key=lambda column: column.str.rstrip("%").astype(float)))

    side_rows = []
    for agent, item in values["agents"].items():
        side_rows.append({
            "Agent": agent,
            "Vai X — W/D/L": wdl(item["as_x"]),
            "Score X": percent(item["as_x"]["score_rate"]),
            "Vai O — W/D/L": wdl(item["as_o"]),
            "Score O": percent(item["as_o"]["score_rate"]),
        })
    display(Markdown("### Tách theo vai"))
    display(pd.DataFrame(side_rows))

    pairs = pd.DataFrame(values["pair_matrix"])[["agent_a", "agent_b", "games", "a_wins", "draws", "b_wins", "a_score_rate", "b_score_rate"]]
    pairs.columns = ["Agent A", "Agent B", "Ván", "A thắng", "Hòa", "B thắng", "Score A", "Score B"]
    pairs["Score A"] = pairs["Score A"].map(percent)
    pairs["Score B"] = pairs["Score B"].map(percent)
    display(Markdown("### Ma trận đối đầu"))
    display(pairs)

## DEFAULT — 1000 ván, 10 cặp, 0 budget violation

### Tổng hợp agent

,Agent,W / D / L,Score,Win CI 95%,Mean move (ms),P95 move (ms),Nodes/move,Sims/move,Seed score SD
1,mcts,274 / 58 / 68,75.75%,63.79–72.86%,30.79,50.24,7.81,6.85,0.286
3,neural,222 / 79 / 99,65.38%,50.60–60.29%,3.66,5.13,0.00,0.00,0.282
0,heuristic,168 / 115 / 117,56.38%,37.26–46.89%,0.47,0.75,75.96,0.00,0.345
2,minimax,178 / 64 / 158,52.50%,39.71–49.40%,37.20,66.48,225.63,0.00,0.331
4,random,0 / 0 / 400,0.00%,0.00–0.95%,0.02,0.04,0.00,0.00,0.000


### Tách theo vai

,Agent,Vai X — W/D/L,Score X,Vai O — W/D/L,Score O
0,heuristic,64 / 105 / 31,58.25%,104 / 10 / 86,54.50%
1,mcts,142 / 32 / 26,79.00%,132 / 26 / 42,72.50%
2,minimax,65 / 7 / 128,34.25%,113 / 57 / 30,70.75%
3,neural,113 / 14 / 73,60.00%,109 / 65 / 26,70.75%
4,random,0 / 0 / 200,0.00%,0 / 0 / 200,0.00%


### Ma trận đối đầu

... (4 outputs left out)

### Tách theo vai

,Agent,Vai X — W/D/L,Score X,Vai O — W/D/L,Score O
0,heuristic,64 / 105 / 31,58.25%,104 / 10 / 86,54.50%
1,mcts,142 / 32 / 26,79.00%,132 / 26 / 42,72.50%
2,minimax,65 / 7 / 128,34.25%,113 / 57 / 30,70.75%
3,neural,113 / 14 / 73,60.00%,109 / 65 / 26,70.75%
4,random,0 / 0 / 200,0.00%,0 / 0 / 200,0.00%


### Ma trận đối đầu

,Agent A,Agent B,Ván,A thắng,Hòa,B thắng,Score A,Score B
0,heuristic,mcts,100,18,15,67,25.50%,74.50%
1,heuristic,minimax,100,50,50,0,75.00%,25.00%
2,heuristic,neural,100,0,50,50,25.00%,75.00%
3,heuristic,random,100,100,0,0,100.00%,0.00%
4,mcts,minimax,100,58,14,28,65.00%,35.00%
5,mcts,neural,100,49,29,22,63.50%,36.50%
6,mcts,random,100,100,0,0,100.00%,0.00%
7,minimax,neural,100,50,0,50,50.00%,50.00%
8,minimax,random,100,100,0,0,100.00%,0.00%
9,neural,random,100,100,0,0,100.00%,0.00%
